# 01 — Data Preprocessing
Center-crops raw images and builds `img_labels.csv` from filenames.
Filename convention: `LOCATION_TOD_WEATHER_N.JPG` (e.g. `LOCUSTWALK_daytime_sunny_1.JPG`)

In [1]:
# ── Colab setup ──────────────────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system pillow pillow-heif pandas
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ── Paths — edit these ───────────────────────────────────────────────────────
if IN_COLAB:
    IMG_FOLDER   = '/content/drive/My Drive/CIS_5190_group_project/Images'
    OUTPUT_DIR   = '/content/drive/My Drive/CIS_5190_group_project/processedImages'
    LABELS_PATH  = '/content/drive/My Drive/CIS_5190_group_project/img_labels.csv'
else:
    IMG_FOLDER   = '../data/raw'
    OUTPUT_DIR   = '../data/processed'
    LABELS_PATH  = '../data/img_labels.csv'

In [5]:
import os, re
import pandas as pd
from PIL import Image
from pillow_heif import register_heif_opener

register_heif_opener()
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_W, TARGET_H = 896, 1194   # smallest dims in the dataset
PATTERN = r'^([^_]+)_([^_]+)_([a-zA-Z]+)'   # location_tod_weather
STEM_PATTERN = r'^([^.]+)\.'

def center_crop(img: Image.Image, tw: int, th: int) -> Image.Image:
    w, h = img.size
    left   = (w - tw) // 2
    top    = (h - th) // 2
    return img.crop((left, top, left + tw, top + th))

existing_df = pd.read_csv(LABELS_PATH) if os.path.exists(LABELS_PATH) else None
already_done_col = 'original_file_name' if existing_df is not None and 'original_file_name' in existing_df.columns else 'file_name'
already_done = set(existing_df[already_done_col].values) if existing_df is not None else set()

schema = {'original_file_name': [], 'file_name': [], 'location': [], 'time_of_day': [], 'weather': []}
badly_named = []

for fname in os.listdir(IMG_FOLDER):
    if fname in already_done:
        continue
    m = re.match(PATTERN, fname)
    if not m:
        badly_named.append(fname)
        continue
    location, tod, weather = m.groups()
    stem = re.match(STEM_PATTERN, fname).group(1)

    in_path  = os.path.join(IMG_FOLDER, fname)
    out_name = stem + '.jpg'
    out_path = os.path.join(OUTPUT_DIR, out_name)

    img = Image.open(in_path).convert('RGB')
    w, h = img.size
    if w >= TARGET_W and h >= TARGET_H:
        img = center_crop(img, TARGET_W, TARGET_H)
    img.save(out_path, 'JPEG')

    schema['original_file_name'].append(fname)
    schema['file_name'].append(out_name)
    schema['location'].append(location)
    schema['time_of_day'].append(tod)
    schema['weather'].append(weather)

new_df = pd.DataFrame(schema)
final_df = pd.concat([existing_df, new_df], ignore_index=True) if existing_df is not None else new_df
final_df.to_csv(LABELS_PATH, index=False)
print(f'Saved {len(final_df)} rows to {LABELS_PATH}')
if badly_named:
    print(f'\nFiles with bad names ({len(badly_named)}):')
    for f in badly_named: print(' ', f)

Saved 109 rows to /content/drive/My Drive/CIS_5190_group_project/img_labels2.csv

Files with bad names (2):
  vp_1.JPG
  vp_2.JPG


In [ ]:
# Quick sanity check
df = pd.read_csv(LABELS_PATH)
print(df.groupby(['time_of_day', 'weather']).size().unstack(fill_value=0))